In [2]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import pandas as pd
import muon as mu
import scanpy as sc
import scirpy as ir
np.random.seed(42)
import random
random.seed(42)

import sys
sys.path.append("/ihome/ylee/yiz133/Code/Data processing/functions")
import importlib
import mdata_utils

2026-05-20 11:00:30.411469: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-20 11:00:30.411854: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-20 11:00:30.444775: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-20 11:00:32.067028: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different compu

In [3]:
# when functions changed
importlib.reload(mdata_utils)

<module 'mdata_utils' from '/ihome/ylee/yiz133/Code/Data processing/functions/mdata_utils.py'>

# GEX

In [4]:
data_path = "/ix1/ylee/Yifan_Zhang/Code_data/external/EAE/GSE188320 Th17"
mdata = mu.read(data_path + "/GSE188320_batch6_9_gex_tcr.h5mu")
mdata

MuData object with n_obs × n_vars = 39845 × 27998
  2 modalities
    gex:	39845 x 27998
      obs:	'batch', 'condition', 'mouse_id', 'assignment_demuxem'
    airr:	39845 x 0
      obsm:	'airr'

In [7]:
mdata['gex'].obs['batch']

AAACCTGAGAGTCGGT-1-b6m1    batch6
AAACCTGAGGCTCAGA-1-b6m1    batch6
AAACCTGCAATAGCAA-1-b6m1    batch6
AAACCTGCAGTCTTCC-1-b6m1    batch6
AAACCTGCATTCTTAC-1-b6m1    batch6
                            ...  
TTTGGTTTCGCTTAGA-1-b9m2    batch9
TTTGTCAAGTATCTCG-1-b9m2    batch9
TTTGTCAGTCGCATCG-1-b9m2    batch9
TTTGTCAGTTTGTGTG-1-b9m2    batch9
TTTGTCATCCTGCTTG-1-b9m2    batch9
Name: batch, Length: 39845, dtype: category
Categories (4, object): ['batch6', 'batch7', 'batch8', 'batch9']

In [5]:
aa

NameError: name 'aa' is not defined

In [ ]:
%cd $data_path

In [ ]:
mdata.obs['GSE'] = 'GSE188320'
mdata.obs['GSM'] = mdata['gex'].obs['batch']
mdata.obs['condition'] = mdata['gex'].obs['condition']

mdata['gex'].obs.rename(columns = {"mouse_id":"sample_id", "assignment_demuxem":"tissue"}, inplace=True)
mdata['gex'].obs['tissue'] = (mdata['gex'].obs['tissue'].replace({'CN': 'CNS', 'SPL': 'Spleen'}))
mdata.obs['tissue'] = mdata['gex'].obs['tissue']

# mdata['gex'].obs['cell_type'] = 'Th17'


In [ ]:
mdata.obs['tissue'].value_counts()

In [ ]:
genes = [g for g in mdata.var_names if g.startswith('Il17')]
genes

In [ ]:
mdata = mdata_utils.pp_EAE(mdata, celltype_score = 0.4, cellstate_score = 0.4, topN_variable = None)

In [ ]:
print(mdata['gex'].obs['cell_type'].value_counts())
print(mdata['gex'].obs['state'].value_counts())

# TCR

In [ ]:
# mdata = mdata_ori.copy()
mdata['airr'].obs['sample_id'] = mdata.obs['sample_id']

ir.pp.index_chains(mdata)
ir.tl.chain_qc(mdata)

ir.pp.ir_dist(mdata)
ir.tl.define_clonotypes(mdata, receptor_arms="all", dual_ir="primary_only", within_group = 'sample_id')

meta_airr = ir.get.airr(mdata['airr'], ["cdr3_aa", "v_call", "j_call"] ,  ('VJ_1', 'VDJ_1'))

df2_filtered = meta_airr.loc[:, ~meta_airr.columns.isin(mdata.obs.columns)]
mdata.obs = mdata.obs.join(df2_filtered)
mdata.update()

# mdata.obs = mdata.obs.reindex(mdata.mod['gex'].obs.index)
ir.tl.clonal_expansion(mdata)


In [ ]:
mdata

In [ ]:
count = (mdata['airr'].obs['clone_id_size'] == 1).sum()
print(count)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
_ = ir.pl.clonal_expansion(
    mdata, 
    target_col="clone_id",
    groupby="cell_type",  # Use the new combined column
    breakpoints=(1, 2, 5), 
    ax = ax
    #normalize=False
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
_ = ir.pl.clonal_expansion(
    mdata, 
    target_col="clone_id",
    groupby="state",  # Use the new combined column
    breakpoints=(1, 2, 5), 
    ax = ax
    #normalize=False
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
_ = ir.pl.clonal_expansion(
    mdata, 
    target_col="clone_id",
    groupby= 'sample_id',  # Use the new combined column
    breakpoints=(1, 2, 5), 
    ax = ax
    #normalize=False
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
_ = ir.pl.clonal_expansion(
    mdata, 
    target_col="clone_id",
    groupby= 'tissue',  # Use the new combined column
    breakpoints=(1, 2, 5), 
    ax = ax
    #normalize=False
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
_ = ir.pl.clonal_expansion(
    mdata, 
    target_col="clone_id",
    groupby= 'condition',  # Use the new combined column
    breakpoints=(1, 2, 5), 
    ax = ax
    #normalize=False
)

In [ ]:
mdata.obs['tissue']

In [ ]:
# mdata.write('yz_processed_allGenes_annotateByRef.h5mu')
mdata.write('yz_processed_allGenes_annotateByScore.h5mu')

In [ ]:
aa

In [ ]:
sc.pp.pca(mdata["gex"], svd_solver="arpack", n_comps=50)
sc.pp.neighbors(mdata["gex"], n_neighbors = 50)
sc.tl.umap(mdata["gex"], min_dist=0.5, spread=1.0)
sc.pl.umap(mdata["gex"], color=['batch', 'sample_id', 'state', 'assignment_demuxem', 'condition'], ncols=2,)